# sec1 - 감사의견

## chromadb 설정

In [33]:
import os, io, json, re, uuid, hashlib, textwrap
from typing import Any, Dict, List, Tuple

# Chroma 초기화 (영속 디렉터리 + 기본 임베딩)
import chromadb
from chromadb.utils import embedding_functions

# CHROMA_PERSIST_DIR:
# - 로컬 디스크에 벡터DB를 영속화합니다.
CHROMA_PERSIST_DIR = os.environ.get("CHROMA_DIR", "./chroma_db")
client = chromadb.PersistentClient(path=CHROMA_PERSIST_DIR)

emb_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="intfloat/multilingual-e5-large"   # <- 여기서 small/base 바꿔주면 됨
)

# 생성/획득할 컬렉션 이름 (sec1 전용)
COLL_NAME = "sec1"

# get_or_create_collection:
# - 존재하면 가져오고, 없으면 생성.
# - embedding_function을 바인딩해 두면 add/query에서 자동 이용
sec1_col = client.get_or_create_collection(name=COLL_NAME, embedding_function=emb_fn)

## 파일 경로 / 입출력 유틸

In [2]:
BASE = "../../json"
INPUT_PATH = os.path.join(BASE, "감사의견.jsonl")

def _strip_bom(s: str) -> str:
    """UTF-8 BOM이 있다면 제거합니다."""
    return s[1:] if s and s[0] == "\ufeff" else s

def read_json_or_jsonl(path: str) -> List[Dict[str, Any]]:
    """
    입력 파일을 JSON Array 또는 JSONL로 유연하게 읽습니다.
    - JSON 배열: [ {...}, {...}, ... ]
    - JSONL: 줄마다 하나의 JSON 객체
    - 공백/주석(//, #) 라인은 무시
    - 끝에 실수로 붙은 콤마(,) 제거
    """
    with open(path, "r", encoding="utf-8") as f:
        raw = _strip_bom(f.read()).strip()

    if not raw:
        return []

    # 1) 배열 JSON인지 먼저 감지
    if raw.startswith("["):
        return json.loads(raw)

    # 2) JSONL 처리
    items = []
    for line in io.StringIO(raw):
        line = line.strip()
        if not line or line.startswith("//") or line.startswith("#"):
            continue
        if line.endswith(","):  # 실수로 남은 trailing comma 제거
            line = line[:-1]
        items.append(json.loads(line))
    return items

def _pick_first(d: Dict[str, Any], keys, default=None):
    """
    키 후보 목록 중 첫 번째로 존재하는 값을 반환.
    - 서로 다른 스키마/표기(영문/국문) 혼용을 흡수하기 위해 사용.
    """
    for k in keys:
        if k in d and d[k] not in (None, ""):
            return d[k]
    return default

## 청크화

In [3]:
def clean_text(s: str) -> str:
    if not s:
        return ""
    s = re.sub(r"(?i)<br\s*/?>", "\n", s)
    s = re.sub(r"(?i)</p\s*>", "\n", s)
    s = re.sub(r"(?i)<p\s*>", "", s)
    s = re.sub(r"<[^>]+>", " ", s)
    s = s.replace("\xa0", " ")
    s = re.sub(r"[ \t\f\v]+", " ", s)
    s = re.sub(r"\s*\n\s*", "\n", s)
    s = re.sub(r"\n{3,}", "\n\n", s)
    return s.strip()

def make_kam_document(rec: Dict[str, Any]) -> str:
    """
    KAM/감사보고 본문을 한 문서로 구성 (헤더+본문)
    - extra_meta 없음: 오직 텍스트만 생성
    """
    company = _pick_first(rec, ["company", "corp", "회사", "issuer", "entity"])
    year    = _pick_first(rec, ["year", "연도", "report_year", "date_year"])
    # 핵심 서술부(있을 수 있는 후보들을 최대한 흡수)
    body_main   = _pick_first(rec, ["kam_text", "kam", "text", "content", "body", "description"])
    body_reason = _pick_first(rec, ["reason", "배경", "핵심감사사항의_선정근거", "why_matter"])
    body_proc   = _pick_first(rec, ["response", "audit_response", "감사인의_대응", "감사절차", "procedures"])

    body_main   = clean_text(str(body_main)   if body_main   is not None else "")
    body_reason = clean_text(str(body_reason) if body_reason is not None else "")
    body_proc   = clean_text(str(body_proc)   if body_proc   is not None else "")

    header = " · ".join([str(x) for x in [company, year] if x])
    sections = []
    if body_main:   sections.append("【핵심감사사항】\n" + body_main)
    if body_reason: sections.append("【선정 사유/배경】\n" + body_reason)
    if body_proc:   sections.append("【감사인의 대응/절차】\n" + body_proc)

    full_text = "\n\n".join([p for p in [header, *sections] if p]).strip()
    return full_text

def chunk_text(text: str, max_chars: int = 1200, overlap: int = 150) -> List[str]:
    text = text.strip()
    if not text:
        return []
    chunks, start, n = [], 0, len(text)
    while start < n:
        end = min(start + max_chars, n)
        if end < n:
            window = text[start:end]
            cut = max(window.rfind("\n"), window.rfind("."))
            if cut >= 200:
                end = start + cut + 1
        chunks.append(text[start:end].strip())
        if end >= n:
            break
        start = max(0, end - overlap)
    return [c for c in chunks if c]

## 메타데이터 빌드

In [4]:
def build_core_metadata(rec: Dict[str, Any], source_path: str) -> Dict[str, Any]:
    """
    오직 core 필드만!
    - company, year, section, section_path, section_name, source, _raw
    """
    company = _pick_first(rec, ["company", "corp", "회사", "issuer", "entity"])
    year    = _pick_first(rec, ["year", "연도", "report_year", "date_year"])

    # section 계층(문서 성격만 표기; downstream에서 필터링 용)
    # KAM/감사의견이 한 파일에 섞여 있을 수 있으므로, record에 명시 없으면 'sec1'로 통일
    section = _pick_first(rec, ["section", "섹션", "kind"]) or "sec1"
    # 보다 명확한 경로/라벨(있으면 쓰고 없으면 기본값)
    section_path = _pick_first(rec, ["section_path", "path"]) or "sec1/auditor_kam"
    section_name = _pick_first(rec, ["section_name", "name", "title"]) or "auditor_kam"

    return {
        "company": company,
        "year": year,
        "section": section,
        "section_path": section_path,
        "section_name": section_name,
        "source": source_path,
        "_raw": json.dumps(rec, ensure_ascii=False, separators=(",", ":")),
    }

def stable_doc_id(company, year, text: str) -> str:
    """extra_meta 없이 company/year/텍스트 일부로 안정 ID 생성"""
    key = f"{company}|{year}|{text[:200]}"
    return "sec1-" + hashlib.md5(key.encode("utf-8")).hexdigest()[:20]

## upsert

In [5]:
def normalize_metadata(md: Dict[str, Any]) -> Dict[str, Any]:
    fixed = {}
    for k, v in md.items():
        if isinstance(v, list):
            # 리스트 → 콤마로 join
            fixed[k] = ", ".join(str(x) for x in v)
        elif isinstance(v, dict):
            # dict → JSON string
            fixed[k] = json.dumps(v, ensure_ascii=False)
        else:
            fixed[k] = v
    return fixed

In [6]:
def upsert_jsonl_direct(input_path: str, collection, batch_size: int = 1000):
    if not os.path.exists(input_path):
        print(f"[WARN] not found: {input_path}")
        return

    items = read_json_or_jsonl(input_path)
    if not items:
        print(f"[WARN] empty: {input_path}")
        return

    batch_ids, batch_docs, batch_mds = [], [], []
    for i, rec in enumerate(items):
        rid = rec.get("id") or f"row-{i}"
        doc = rec.get("document", "").strip()
        md  = rec.get("metadata", {})

        if not doc:
            continue

        # ⚡️ metadata 정규화
        md = normalize_metadata(md)

        batch_ids.append(rid)
        batch_docs.append(doc)
        batch_mds.append(md)

        if len(batch_ids) >= batch_size:
            collection.upsert(ids=batch_ids, documents=batch_docs, metadatas=batch_mds)
            batch_ids, batch_docs, batch_mds = [], [], []

    if batch_ids:
        collection.upsert(ids=batch_ids, documents=batch_docs, metadatas=batch_mds)

    print(f"[OK] {len(items)} records upserted into collection '{collection.name}'")

In [7]:
upsert_jsonl_direct(INPUT_PATH, sec1_col, batch_size=1000)

[OK] 26 records upserted into collection 'sec1'


## 쿼리 테스트

### 간단한 질의

In [8]:
import re

def extract_year(text: str) -> int | None:
    """
    질의문에서 4자리 'YYYY년' 혹은 'YYYY' 패턴을 찾아서 반환
    """
    m = re.search(r"(20\d{2}|19\d{2})년?", text)
    if m:
        return int(m.group(1))
    return None

def query_with_year_filter(collection, query_text: str, top_k: int = 3):
    # 연도 추출
    year = extract_year(query_text)

    where_clause = None
    if year:
        where_clause = {"year": year}

    # 쿼리 실행
    res = collection.query(
        query_texts=[query_text],
        n_results=top_k,
        include=["documents", "metadatas", "distances"],
        where=where_clause
    )

    # 결과 출력
    for doc, md, dist in zip(res["documents"][0], res["metadatas"][0], res["distances"][0]):
        print("="*80)
        print(f"dist={dist:.4f}")
        print("문서:", (doc[:200] + "...") if len(doc) > 200 else doc)
        print("메타:", md)

# 사용 예시
query_with_year_filter(sec1_col, "2019년 주요감사사항은?", top_k=3)

dist=0.1702
문서: 핵심감사사항 #2: 감가상각 개시시점의 적절성. [이유] 2019년 12월 31일 현재 삼성전자는 총 74090275 백만원의 유형자산을 보유하고 있으며, 해당 유형자산은 건물 및 기계장치 등을 포함하고 있습니다. 이 중 당기 중 취득으로 인한 증가금액은 18210158 백만원입니다(주석 12 참조). 삼성전자는 라인 및 설비의 사용가능시점을 판단하여 각 ...
메타: {'parent_id': 'sec1-2019-root', 'group_id': 'sec1-2019-root', 'year': 2019, 'year_anchor': '삼성전자-2019', 'reason': '2019년 12월 31일 현재 삼성전자는 총 74090275 백만원의 유형자산을 보유하고 있으며, 해당 유형자산은 건물 및 기계장치 등을 포함하고 있습니다. 이 중 당기 중 취득으로 인한 증가금액은 18210158 백만원입니다(주석 12 참조). 삼성전자는 라인 및 설비의 사용가능시점을 판단하여 각 자산에 대한 감가상각을 개시합니다(주석 2.9 참조).\n삼성전자의 투자규모가 크고 사용가능시점 판단에 따른 감가상각금액이 재무제표에 미치는 영향이 유의적이므로 삼일회계법인는 감가상각 개시시점의 적절성 관련 회계처리를 핵심감사사항으로 판단하였습니다.', 'company_anchor': '삼성전자', 'source': '감사보고서_2019_preprocess.html#SEC1', 'note_doc_ids': 'sec3-2019-note-12, sec3-2019-note-2.9', 'title': '감가상각 개시시점의 적절성', 'kam_index': 2, 'section': 'SEC1', 'kind': 'kam', 'company': '삼성전자', 'note_refs': '12, 2.9'}
dist=0.1712
문서: 2019년 삼성전자 재무제표의 외부감사는 삼일회계법인이 수행했습니다.
메타: {'company': '삼성전자', 'year': 2019, 'com

### DB 구조 확인

In [9]:
print("== 컬렉션 목록 ==")
for c in client.list_collections():
    print("-", c.name)

# sec1 컬렉션 정보
print("\n== sec1 info ==")
info = sec1_col.get(limit=2)  # 샘플 2개만
print("id:", info["ids"])
print("문서:", info["documents"])
print("메타:", info["metadatas"])

== 컬렉션 목록 ==
- sec1

== sec1 info ==
id: ['sec1-2014-auditor', 'sec1-2014-kam-none']
문서: ['2014년 삼성전자 재무제표의 외부감사는 삼일회계법인이 수행했습니다.', '2014년은 핵심감사사항 제도 도입 이전으로 핵심감사사항이 없습니다.']
메타: [{'year_anchor': '삼성전자-2014', 'parent_id': 'sec1-2014-root', 'group_id': 'sec1-2014-root', 'section': 'SEC1', 'source': '감사보고서_2014_preprocess.html#SEC1', 'kind': 'auditor', 'company': '삼성전자', 'year': 2014, 'company_anchor': '삼성전자'}, {'source': '감사보고서_2014_preprocess.html#SEC1', 'group_id': 'sec1-2014-root', 'company': '삼성전자', 'company_anchor': '삼성전자', 'year': 2014, 'section': 'SEC1', 'parent_id': 'sec1-2014-root', 'kind': 'kam_none', 'year_anchor': '삼성전자-2014'}]


# sec2 - 재무상태표, 손익계산서

In [10]:
COLL_NAME = "sec2"
sec2_col = client.get_or_create_collection(name=COLL_NAME, embedding_function=emb_fn)

In [11]:
upsert_jsonl_direct("../../json/손익계산서_v2.jsonl", sec2_col)

# 샘플 확인
info = sec2_col.get(limit=2)
print("샘플 id:", info["ids"])
print("샘플 문서:", info["documents"])
print("샘플 메타:", info["metadatas"])

[OK] 154 records upserted into collection 'sec2'
샘플 id: ['sec2-2014-is-매출액-매출액', 'sec2-2014-is-매출원가-매출원가']
샘플 문서: ['2014년 삼성전자의 매출액은(는) 137,825,547 백만원입니다(전기 158,372,089 백만원).', '2014년 삼성전자의 매출원가은(는) 99,188,713 백만원입니다(전기 110,731,528 백만원).']
샘플 메타: [{'line_item': '매출액', 'statement_code': 'is', 'path_labels': '매출액', 'parent_id': 'sec2-2014-is-node-매출액', 'section': 'SEC2', 'unit': 'KRW_million', 'group_id': 'sec2-2014-is-root', 'note_refs': '', 'statement_anchor': '삼성전자-2014-is', 'year': 2014, 'path_ord': '1, 1', 'note_doc_ids': '', 'year_anchor': '삼성전자-2014', 'company': '삼성전자', 'path_codes': '매출액', 'statement': '손익계산서'}, {'path_labels': '매출원가', 'path_codes': '매출원가', 'line_item': '매출원가', 'statement_code': 'is', 'year': 2014, 'path_ord': '1, 2', 'statement_anchor': '삼성전자-2014-is', 'parent_id': 'sec2-2014-is-node-매출원가', 'statement': '손익계산서', 'note_doc_ids': 'sec3-2014-note-25', 'year_anchor': '삼성전자-2014', 'group_id': 'sec2-2014-is-root', 'section': 'SEC2', 'company': '삼성전자', 'unit': 'KRW_mi

In [12]:
upsert_jsonl_direct("../../json/재무상태표.jsonl", sec2_col)

[OK] 501 records upserted into collection 'sec2'


In [13]:
upsert_jsonl_direct("../../json/현금흐름표.jsonl", sec2_col)
upsert_jsonl_direct("../../json/자본변동표_dedup.jsonl", sec2_col)
upsert_jsonl_direct("../../json/포괄손익계산서.jsonl", sec2_col)

[OK] 346 records upserted into collection 'sec2'
[OK] 265 records upserted into collection 'sec2'
[OK] 33 records upserted into collection 'sec2'


In [14]:
query_text = "2019년 매출은 얼마야?"
res = query_with_year_filter(sec2_col, query_text, top_k=3)

dist=0.1512
문서: 2019년 삼성전자의 매출액은(는) 154,772,859 백만원입니다.
메타: {'path_codes': '매출액', 'year': 2019, 'group_id': 'sec2-2019-is-root', 'section': 'SEC2', 'note_refs': '31', 'line_item': '매출액', 'year_anchor': '삼성전자-2019', 'path_labels': '매출액', 'note_doc_ids': 'sec3-2019-note-31', 'path_ord': '1, 1', 'statement_code': 'is', 'parent_id': 'sec2-2019-is-node-매출액', 'unit': 'KRW_million', 'statement': '손익계산서', 'company': '삼성전자', 'statement_anchor': '삼성전자-2019-is'}
dist=0.1520
문서: 2019년 삼성전자의 매출원가은(는) 113,618,444 백만원입니다.
메타: {'group_id': 'sec2-2019-is-root', 'statement_code': 'is', 'statement': '손익계산서', 'path_ord': '1, 2', 'path_codes': '매출원가', 'statement_anchor': '삼성전자-2019-is', 'note_doc_ids': 'sec3-2019-note-23', 'year': 2019, 'path_labels': '매출원가', 'year_anchor': '삼성전자-2019', 'parent_id': 'sec2-2019-is-node-매출원가', 'section': 'SEC2', 'unit': 'KRW_million', 'note_refs': '23', 'line_item': '매출원가', 'company': '삼성전자'}
dist=0.1525
문서: 2019년 삼성전자의 매출총이익은(는) 41,154,415 백만원입니다.
메타: {'path_ord': '1, 3', '

# QA 성능평가

In [15]:
import re, textwrap
from typing import Dict, Any, List

COL_NAMES = ["sec1", "sec2"]
collections = {name: client.get_collection(name=name, embedding_function=emb_fn) for name in COL_NAMES}

In [16]:
import json
import textwrap

# --------------------------------------------------------
# QA 파일 로더
def read_qa(path: str):
    import io
    def _strip_bom(s: str) -> str:
        return s[1:] if s and s[0] == "\ufeff" else s
    with open(path, "r", encoding="utf-8") as f:
        raw = _strip_bom(f.read()).strip()
    if raw.startswith("["):
        return json.loads(raw)
    items = []
    for line in io.StringIO(raw):
        line = line.strip()
        if not line or line.startswith("//") or line.startswith("#"):
            continue
        if line.endswith(","):
            line = line[:-1]
        items.append(json.loads(line))
    return items


# --------------------------------------------------------
# Reciprocal Rank 계산 (여러 정답 평균)
def reciprocal_rank_avg(ranked_ids, gold_ids):
    ranks = []
    for g in gold_ids:
        if g in ranked_ids:
            r = ranked_ids.index(g) + 1   # 1-based rank
            ranks.append(1.0 / r)
    if not ranks:
        return 0.0
    return sum(ranks) / len(ranks)

# 전체 MRR_avg 평가 (scores도 반환)
def eval_mrr_avg(qa_path: str, top_k: int = 10, max_samples: int = None):
    qa_items = read_qa(qa_path)
    if max_samples:
        qa_items = qa_items[:max_samples]

    scores = []
    for idx, item in enumerate(qa_items, 1):
        qid   = item.get("id")
        qtext = item.get("question", "")
        golds = item.get("answers", [])

        merged = []
        for cname in ["sec1", "sec2"]:
            try:
                res = collections[cname].query(
                    query_texts=[qtext],
                    n_results=top_k,
                    include=["documents", "metadatas", "distances"],  # ✅ ids는 자동 반환
                )
                for doc_id, dist in zip(res["ids"][0], res["distances"][0]):
                    merged.append((dist, doc_id))
            except Exception as e:
                print(f"[{qid}] query error on {cname}: {e}")

        # 거리 기준 정렬 후 상위 top_k
        merged.sort(key=lambda x: x[0])
        ranked_ids = [doc_id for _, doc_id in merged[:top_k]]

        rr_avg = reciprocal_rank_avg(ranked_ids, golds)
        scores.append(rr_avg)

        print(f"[{idx}] {qid} | RR_avg={rr_avg:.4f} | Q='{qtext}'")
        print(f"   Gold: {golds}")
        print(f"   Top-{top_k}: {ranked_ids}")
        print()

    mrr_avg = sum(scores) / len(scores) if scores else 0.0
    print("="*80)
    print(f"MRR_avg (over {len(scores)} questions, top-{top_k}) = {mrr_avg:.4f}")

    # ✅ 이제 (평균, 점수 리스트) 둘 다 반환
    return mrr_avg, scores

# --------------------------------------------------------
# 실행 예시
QA_PATH = "../../qa_dataset_50_revised.jsonl"
mrr_value, scores = eval_mrr_avg(QA_PATH, top_k=10, max_samples=50)

[1] Q001 | RR_avg=1.0000 | Q='2019년 감사보고서를 담당한 회계 감사 기관은 어디인가?'
   Gold: ['sec1-2019-auditor']
   Top-10: ['sec1-2019-auditor', 'sec1-2020-auditor', 'sec1-2024-auditor', 'sec1-2023-auditor', 'sec1-2018-auditor', 'sec1-2014-auditor', 'sec1-2022-auditor', 'sec1-2016-auditor', 'sec1-2021-auditor', 'sec1-2017-auditor']

[2] Q002 | RR_avg=1.0000 | Q='2020년 감사를 담당한 곳은?'
   Gold: ['sec1-2020-auditor']
   Top-10: ['sec1-2020-auditor', 'sec1-2024-auditor', 'sec1-2021-auditor', 'sec1-2019-auditor', 'sec1-2023-auditor', 'sec1-2022-auditor', 'sec1-2014-auditor', 'sec1-2016-auditor', 'sec1-2015-auditor', 'sec1-2017-auditor']

[3] Q003 | RR_avg=0.0000 | Q='2023년 핵심감사사항은?'
   Gold: ['sec1-2023-kam-1']
   Top-10: ['sec1-2017-kam-none', 'sec1-2015-kam-none', 'sec1-2023-auditor', 'sec1-2016-kam-none', 'sec1-2019-kam-2', 'sec1-2018-kam-2', 'sec1-2014-kam-none', 'sec1-2024-auditor', 'sec1-2024-kam-1', 'sec1-2018-kam-1']

[4] Q004 | RR_avg=1.0000 | Q='2018년 재화의 판매 관련 매출장려활동이 핵심감사사항으로 결정된 이유는?'
   Gold: ['s

In [17]:
import csv

output_csv = "../result/e5_raw_scores.csv"
with open(output_csv, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["QuestionID", "RR_avg"])
    for idx, score in enumerate(scores, 1):
        writer.writerow([f"Q{idx:03d}", score])

## json에서 line_item 추출 & 저장

In [18]:
import os, io, json

JSON_DIR = '../../json/'

def _strip_bom(s: str) -> str:
    return s[1:] if s and s[0] == "\ufeff" else s

def read_json_or_jsonl(path: str):
    """JSON/JSONL 모두 유연하게 읽기"""
    with open(path, "r", encoding="utf-8") as f:
        raw = _strip_bom(f.read()).strip()
    if not raw:
        return []
    if raw.startswith("["):
        return json.loads(raw)
    items = []
    for line in io.StringIO(raw):
        line = line.strip()
        if not line or line.startswith("//") or line.startswith("#"):
            continue
        if line.endswith(","):
            line = line[:-1]
        items.append(json.loads(line))
    return items

def collect_line_items(json_dir: str) -> set[str]:
    line_items = set()
    for fname in os.listdir(json_dir):
        if not fname.endswith(".jsonl"):
            continue
        fpath = os.path.join(json_dir, fname)
        items = read_json_or_jsonl(fpath)
        for rec in items:
            md = rec.get("metadata", {})
            li = md.get("line_item")
            if li:
                line_items.add(li.strip())
    return line_items

all_line_items = collect_line_items(JSON_DIR)

In [27]:
import re
import io
import json
import textwrap
from typing import List, Dict, Any, Optional
from FlagEmbedding import FlagReranker

# --------------------------------------------------------
# QA 파일 로더 (기존과 동일)
def read_qa(path: str) -> List[Dict[str, Any]]:
    def _strip_bom(s: str) -> str:
        return s[1:] if s and s[0] == "\ufeff" else s
    with open(path, "r", encoding="utf-8") as f:
        raw = _strip_bom(f.read()).strip()
    if raw.startswith("["):
        return json.loads(raw)
    items = []
    for line in io.StringIO(raw):
        line = line.strip()
        if not line or line.startswith("//") or line.startswith("#"):
            continue
        if line.endswith(","):
            line = line[:-1]
        items.append(json.loads(line))
    return items

# --------------------------------------------------------
# 연도/상대연도 파싱 (기존과 동일)
def extract_years_with_relative(text: str) -> list[int]:
    years = set()
    for y in re.findall(r"(19\d{2}|20\d{2})", text):
        years.add(int(y))
    base = max(years) if years else None
    if base:
        if any(x in text for x in ["전년","작년","전기","전분기"]):
            years.add(base - 1)
        if any(x in text for x in ["재작년","전전기","전전분기"]):
            years.add(base - 2)
        if any(x in text for x in ["올해","금년"]):
            years.add(base)
    return sorted(list(years))

# --------------------------------------------------------
# 질문에서 line_item 추출 (사전 처리 로직 추가)
try:
    LINE_ITEM_KEYWORDS = list(all_line_items) # set일 수 있으므로 list로 변환
except NameError:
    LINE_ITEM_KEYWORDS = []

LINE_ITEM_KEYWORDS.sort(key=len, reverse=True)
LINE_ITEM_KEYWORDS_NO_SPACE = [kw.replace(" ", "") for kw in LINE_ITEM_KEYWORDS]
LINE_ITEM_PAIRS = list(zip(LINE_ITEM_KEYWORDS, LINE_ITEM_KEYWORDS_NO_SPACE))

def extract_line_item(text: str) -> Optional[str]:
    text_no_space = text.replace(" ", "")
    for original_kw, kw_no_space in LINE_ITEM_PAIRS:
        if kw_no_space in text_no_space:
            return original_kw
    return None

# --------------------------------------------------------
# 리랭커 준비 (기존과 동일)
reranker = FlagReranker("BAAI/bge-reranker-v2-m3", use_fp16=True)

# --------------------------------------------------------
# Reciprocal Rank 계산 (기존과 동일)
def reciprocal_rank_avg(ranked_ids: list[str], gold_ids: list[str]) -> float:
    ranks = []
    for g in gold_ids:
        if g in ranked_ids:
            r = ranked_ids.index(g) + 1
            ranks.append(1.0 / r)
    if not ranks:
        return 0.0
    return sum(ranks) / len(gold_ids)

# --------------------------------------------------------
# ✅ (수정) 메인 평가 함수: 검색 로직 통합
def eval_mrr_avg_with_boosting(qa_path: str, max_samples: int = 50, top_k: int = 10, dense_k: int = 15):
    qa_items = read_qa(qa_path)
    total = len(qa_items)
    print(f"총 {total}개 QA 중 {max_samples}개 샘플 평가\n")

    scores = []
    for idx, item in enumerate(qa_items[:max_samples], 1):
        qid    = item.get("id")
        qtext  = item.get("question", "")
        golds  = item.get("answers", [])

        print("="*100)
        print(f"[{idx}] ID={qid}")
        print(f"Q: {qtext}")
        print(f"Gold IDs: {golds}")

        years = extract_years_with_relative(qtext)
        line_item = extract_line_item(qtext)

        candidates: list[tuple[str, str, dict]] = []
        
        # ✅ 쿼리 확장: line_item이 있으면 검색어에 추가하여 정확도 향상
        queries = [qtext]
        if line_item and years:
            for y in years:
                queries.append(f"{y}년 {line_item}")

        # ✅ 검색 로직 통합: line_item 유무에 관계없이 sec1, sec2 모두 검색
        for sub_q in queries:
            for cname in ["sec1", "sec2"]:
                try:
                    res = collections[cname].query(
                        query_texts=[sub_q],
                        n_results=dense_k,
                        include=["documents", "metadatas"],
                        where={"year": {"$in": years}} if years else None,
                    )
                    docs = res.get("documents", [[]])[0]
                    mds  = res.get("metadatas", [[]])[0]
                    ids  = res.get("ids", [[]])[0]
                    for doc, md, doc_id in zip(docs, mds, ids):
                        if not all([doc, md, doc_id]): continue
                        candidates.append((doc[:2000], doc_id, md))
                except Exception as e:
                    print(f"[WARN] {cname} '{sub_q}' query error: {e}")

        # 중복 후보 제거 (ID 기준)
        unique_candidates = []
        seen_ids = set()
        for doc, doc_id, md in candidates:
            if doc_id not in seen_ids:
                unique_candidates.append((doc, doc_id, md))
                seen_ids.add(doc_id)

        ranked_ids = []
        if unique_candidates:
            # 리랭커 점수 계산
            rerank_candidates = [(doc, doc_id) for doc, doc_id, md in unique_candidates]
            pairs = [[qtext, c[0]] for c in rerank_candidates]
            scores_reranker = reranker.compute_score(pairs, normalize=True, batch_size=32)

            # 하이브리드 리랭킹: line_item 일치 시 보너스 점수 부여
            final_ranked_items = []
            for (doc_text, doc_id, md), score in zip(unique_candidates, scores_reranker):
                bonus = 0.0
                if line_item and md.get("line_item") == line_item:
                    bonus = 0.1
                
                final_ranked_items.append({
                    "id": doc_id,
                    "final_score": score + bonus
                })
            
            final_ranked_items.sort(key=lambda x: x["final_score"], reverse=True)
            ranked_ids = [item["id"] for item in final_ranked_items[:top_k]]

        rr_avg = reciprocal_rank_avg(ranked_ids, golds)
        scores.append(rr_avg)

        print(f" → RR_avg={rr_avg:.4f}")
        print(f"   Ranked IDs: {ranked_ids}")
        print()

    mrr_avg = sum(scores) / len(scores) if scores else 0.0
    print("="*80)
    print(f"MRR_avg (over {len(scores)} questions, top-{top_k}) = {mrr_avg:.4f}")

    return mrr_avg, scores

# --------------------------------------------------------
# 실행
QA_PATH = "../../qa_dataset_50_revised.jsonl"
mrr_value, scores = eval_mrr_avg_with_boosting(QA_PATH, max_samples=50, top_k=10, dense_k=8)

총 50개 QA 중 50개 샘플 평가

[1] ID=Q001
Q: 2019년 감사보고서를 담당한 회계 감사 기관은 어디인가?
Gold IDs: ['sec1-2019-auditor']


You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


 → RR_avg=1.0000
   Ranked IDs: ['sec1-2019-auditor', 'sec1-2019-kam-2', 'sec1-2019-kam-1', 'sec2-2019-se-총포괄손익-3-순확정급여자산재측정요소-기타자본항목-b6143e85', 'sec2-2019-se-총포괄손익-2-기타포괄손익공정가치금융자산평가손익-기타자본항목-f5ba8a2d', 'sec2-2019-se-총포괄손익-3-순확정급여자산재측정요소-총계-4da696b1', 'sec2-2019-cf-현금흐름-투자활동현금흐름-4-기타포괄손익공정가치금융자산의처분', 'sec2-2019-is-법인세비용차감전순이익-법인세비용', 'sec2-2019-is-법인세비용차감전순이익-법인세비용차감전순이익', 'sec2-2019-bs-부채-유동부채-5-예수금']

[2] ID=Q002
Q: 2020년 감사를 담당한 곳은?
Gold IDs: ['sec1-2020-auditor']
 → RR_avg=1.0000
   Ranked IDs: ['sec1-2020-auditor', 'sec1-2020-kam-1', 'sec2-2020-cf-현금흐름-재무활동현금흐름-3-배당금의지급', 'sec2-2020-cf-현금흐름-영업활동현금흐름-2-이자의수취', 'sec2-2020-ci-기타포괄손익', 'sec2-2020-cf-현금흐름-투자활동현금흐름-3-기타포괄손익공정가치금융자산의처분', 'sec2-2020-cf-현금흐름-투자활동현금흐름-4-기타포괄손익공정가치금융자산의취득', 'sec2-2020-bs-자본-기타자본항목-기타자본항목', 'sec2-2020-is-영업이익-기타수익', 'sec2-2020-bs-부채-유동부채-4-선수금']

[3] ID=Q003
Q: 2023년 핵심감사사항은?
Gold IDs: ['sec1-2023-kam-1']
 → RR_avg=1.0000
   Ranked IDs: ['sec1-2023-kam-1', 'sec1-2023-kam-2', 'sec1-2023-auditor', 'sec2-2023-c

In [28]:
import csv

output_csv = "../result/e5_scores.csv"
with open(output_csv, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["QuestionID", "RR_avg"])
    for idx, score in enumerate(scores, 1):
        writer.writerow([f"Q{idx:03d}", score])

In [37]:
import re
import io
import json
from typing import List, Dict, Any, Optional

# 필요한 라이브러리 임포트
from FlagEmbedding import FlagReranker
from rank_bm25 import BM25Okapi
from konlpy.tag import Okt
import chromadb
from chromadb.utils import embedding_functions # 예시를 위해 추가

# --------------------------------------------------------
# 0. ChromaDB 및 BM25 인덱스 초기화
# --------------------------------------------------------
print("사전 준비 작업을 시작합니다...")

# --- ChromaDB 클라이언트 및 컬렉션 로드 ---
try:
    print(f"기존 ChromaDB 컬렉션: {list(collections.keys())}를 사용합니다.")
except NameError:
    print("ChromaDB `collections` 변수를 찾을 수 없어, 예시를 생성합니다.")
    CHROMA_PERSIST_DIR = "./chroma_db"
    client = chromadb.PersistentClient(path=CHROMA_PERSIST_DIR)
    # 실제 사용하는 임베딩 모델 이름으로 변경해야 합니다.
    emb_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
        model_name="intfloat/multilingual-e5-large"
    )
    collections = {
        "sec1": client.get_collection(name="sec1", embedding_function=emb_fn),
        "sec2": client.get_collection(name="sec2", embedding_function=emb_fn)
    }

# --- BM25 인덱스 생성을 위한 전체 문서 로딩 ---
all_docs_for_bm25 = []
id_to_doc_text_map = {}
id_to_metadata_map = {}

print("BM25 인덱스를 위해 ChromaDB에서 모든 문서를 로드합니다.")
for cname in ["sec1", "sec2"]:
    results = collections[cname].get(include=["documents", "metadatas"])
    for doc_id, doc_text, md in zip(results['ids'], results['documents'], results['metadatas']):
        all_docs_for_bm25.append({'id': doc_id, 'text': doc_text})
        id_to_doc_text_map[doc_id] = doc_text
        id_to_metadata_map[doc_id] = md

bm25_doc_texts = [doc['text'] for doc in all_docs_for_bm25]
bm25_doc_ids = [doc['id'] for doc in all_docs_for_bm25]

print("BM25 인덱싱을 시작합니다. 문서 양에 따라 시간이 걸릴 수 있습니다.")
okt = Okt()
tokenized_corpus = [okt.morphs(doc) for doc in bm25_doc_texts]
bm25_index = BM25Okapi(tokenized_corpus)
print("BM25 인덱스 생성이 완료되었습니다.")


# --------------------------------------------------------
# 1. 유틸리티 함수 정의
# --------------------------------------------------------

def read_qa(path: str) -> List[Dict[str, Any]]:
    def _strip_bom(s: str) -> str:
        return s[1:] if s and s[0] == "\ufeff" else s
    with open(path, "r", encoding="utf-8") as f:
        raw = _strip_bom(f.read()).strip()
    if raw.startswith("["):
        return json.loads(raw)
    items = []
    for line in io.StringIO(raw):
        line = line.strip()
        if not line or line.startswith("//") or line.startswith("#"): continue
        if line.endswith(","): line = line[:-1]
        items.append(json.loads(line))
    return items

def extract_years_with_relative(text: str) -> list[int]:
    years = set()
    range_patterns = [
        r"(\d{4})\s*년\s*부터\s*(\d{4})\s*년",
        r"(\d{4})\s*[~-]\s*(\d{4})"
    ]
    processed_text = text
    for pattern in range_patterns:
        for match in re.finditer(pattern, processed_text):
            start_year, end_year = int(match.group(1)), int(match.group(2))
            if start_year <= end_year:
                years.update(range(start_year, end_year + 1))
            processed_text = processed_text.replace(match.group(0), "")
    for y in re.findall(r"(19\d{2}|20\d{2})", processed_text):
        years.add(int(y))
    if not years: return []
    base = max(years)
    if any(x in text for x in ["전년","작년","전기"]): years.add(base - 1)
    if any(x in text for x in ["재작년","전전기"]): years.add(base - 2)
    return sorted(list(years))

try:
    all_line_items = set(md['line_item'] for md in id_to_metadata_map.values() if md.get('line_item'))
    LINE_ITEM_KEYWORDS = list(all_line_items)
except NameError:
    LINE_ITEM_KEYWORDS = []
LINE_ITEM_KEYWORDS.sort(key=len, reverse=True)
LINE_ITEM_KEYWORDS_NO_SPACE = [kw.replace(" ", "") for kw in LINE_ITEM_KEYWORDS]
LINE_ITEM_PAIRS = list(zip(LINE_ITEM_KEYWORDS, LINE_ITEM_KEYWORDS_NO_SPACE))
def extract_line_item(text: str) -> Optional[str]:
    text_no_space = text.replace(" ", "")
    for original_kw, kw_no_space in LINE_ITEM_PAIRS:
        if kw_no_space in text_no_space:
            return original_kw
    return None

print("리랭커 모델을 로드합니다...")
reranker = FlagReranker("BAAI/bge-reranker-base", use_fp16=True)
print("리랭커 모델 로드가 완료되었습니다.")

def reciprocal_rank_avg(ranked_ids: list[str], gold_ids: list[str]) -> float:
    ranks = []
    for g in gold_ids:
        if g in ranked_ids:
            r = ranked_ids.index(g) + 1
            ranks.append(1.0 / r)
    if not ranks: return 0.0
    return sum(ranks) / len(gold_ids)

# --------------------------------------------------------
# 2. 하이브리드 검색 함수 정의 (No Filter)
# --------------------------------------------------------
def hybrid_search(query: str, dense_k: int, k_rrf: int = 60) -> List[str]:
    # --- 2.1 키워드 검색 (BM25) ---
    tokenized_query = okt.morphs(query)
    bm25_scores = bm25_index.get_scores(tokenized_query)
    bm25_results = sorted(zip(bm25_doc_ids, bm25_scores), key=lambda x: x[1], reverse=True)[:dense_k]

    # --- 2.2 벡터 검색 (ChromaDB) ---
    vector_results_ids = []
    for cname in ["sec1", "sec2"]:
        res = collections[cname].query(query_texts=[query], n_results=dense_k, where=None)
        vector_results_ids.extend(res['ids'][0])
    
    # --- 2.3 결과 융합 (RRF) ---
    scores = {}
    
    for rank, (doc_id, score) in enumerate(bm25_results, 1):
        if doc_id not in scores: scores[doc_id] = 0.0
        scores[doc_id] += 1 / (k_rrf + rank)

    unique_vector_ids = list(dict.fromkeys(vector_results_ids)) # 순서 유지하며 중복 제거
    for rank, doc_id in enumerate(unique_vector_ids, 1):
        if doc_id not in scores: scores[doc_id] = 0.0
        scores[doc_id] += 1 / (k_rrf + rank)
        
    fused_results = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    
    return [doc_id for doc_id, score in fused_results[:dense_k]]

# --------------------------------------------------------
# 3. 메인 평가 함수
# --------------------------------------------------------
def eval_mrr_hybrid_boosted(qa_path: str, max_samples: int = 50, top_k: int = 10, dense_k: int = 20):
    qa_items = read_qa(qa_path)
    print(f"\n총 {len(qa_items)}개 QA 중 {max_samples}개 샘플 평가 (Hybrid Search + Rerank + Boosting)")

    scores = []
    for idx, item in enumerate(qa_items[:max_samples], 1):
        qid, qtext, golds = item.get("id"), item.get("question", ""), item.get("answers", [])
        print("="*100 + f"\n[{idx}] ID={qid}\nQ: {qtext}\nGold IDs: {golds}")

        years = extract_years_with_relative(qtext)
        line_item = extract_line_item(qtext)
        
        retrieved_ids = hybrid_search(qtext, dense_k=dense_k)
        
        unique_candidates = []
        for doc_id in retrieved_ids:
            doc_text = id_to_doc_text_map.get(doc_id, "")
            md = id_to_metadata_map.get(doc_id, {})
            if doc_text and md:
                unique_candidates.append((doc_text[:2000], doc_id, md))

        ranked_ids = []
        if unique_candidates:
            rerank_candidates = [(doc, doc_id) for doc, doc_id, md in unique_candidates]
            pairs = [[qtext, c[0]] for c in rerank_candidates]
            scores_reranker = reranker.compute_score(pairs, normalize=True, batch_size=32)
            
            final_ranked_items = []
            for (doc_text, doc_id, md), score in zip(unique_candidates, scores_reranker):
                bonus = 0.0
                if line_item and md.get("line_item") == line_item:
                    bonus += 0.1
                if years and md.get("year") in years:
                    bonus += 0.1
                
                final_ranked_items.append({"id": doc_id, "final_score": score + bonus})
            
            final_ranked_items.sort(key=lambda x: x["final_score"], reverse=True)
            ranked_ids = [item["id"] for item in final_ranked_items[:top_k]]

        rr_avg = reciprocal_rank_avg(ranked_ids, golds)
        scores.append(rr_avg)

        print(f" → RR_avg={rr_avg:.4f}\n   Ranked IDs: {ranked_ids}\n")

    mrr_avg = sum(scores) / len(scores) if scores else 0.0
    print("="*80 + f"\nMRR_avg (over {len(scores)} questions, top-{top_k}) = {mrr_avg:.4f}")
    return mrr_avg, scores

# --------------------------------------------------------
# 4. 실행
# --------------------------------------------------------
QA_PATH = "../../qa_dataset_50_revised.jsonl"
mrr_value, all_scores = eval_mrr_hybrid_boosted(QA_PATH, max_samples=50, top_k=10, dense_k=20)

사전 준비 작업을 시작합니다...
기존 ChromaDB 컬렉션: ['sec1', 'sec2']를 사용합니다.
BM25 인덱스를 위해 ChromaDB에서 모든 문서를 로드합니다.
BM25 인덱싱을 시작합니다. 문서 양에 따라 시간이 걸릴 수 있습니다.
BM25 인덱스 생성이 완료되었습니다.
리랭커 모델을 로드합니다...
리랭커 모델 로드가 완료되었습니다.

총 50개 QA 중 50개 샘플 평가 (Hybrid Search + Rerank + Boosting)
[1] ID=Q001
Q: 2019년 감사보고서를 담당한 회계 감사 기관은 어디인가?
Gold IDs: ['sec1-2019-auditor']


You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


 → RR_avg=0.5000
   Ranked IDs: ['sec1-2019-kam-2', 'sec1-2019-auditor', 'sec1-2023-kam-1', 'sec1-2024-kam-1', 'sec1-2021-kam-1', 'sec1-2021-auditor', 'sec1-2020-auditor', 'sec1-2018-kam-2', 'sec1-2017-auditor', 'sec1-2016-auditor']

[2] ID=Q002
Q: 2020년 감사를 담당한 곳은?
Gold IDs: ['sec1-2020-auditor']
 → RR_avg=1.0000
   Ranked IDs: ['sec1-2020-auditor', 'sec2-2020-is-매출원가-매출원가', 'sec1-2024-kam-1', 'sec1-2023-kam-1', 'sec1-2021-auditor', 'sec1-2016-auditor', 'sec1-2018-auditor', 'sec1-2015-auditor', 'sec1-2017-auditor', 'sec1-2014-auditor']

[3] ID=Q003
Q: 2023년 핵심감사사항은?
Gold IDs: ['sec1-2023-kam-1']
 → RR_avg=1.0000
   Ranked IDs: ['sec1-2023-kam-1', 'sec1-2021-kam-1', 'sec1-2022-kam-1', 'sec1-2020-kam-1', 'sec1-2018-kam-1', 'sec1-2024-kam-1', 'sec1-2019-kam-1', 'sec1-2023-kam-2', 'sec1-2023-auditor', 'sec2-2023-is-매출원가-매출원가']

[4] ID=Q004
Q: 2018년 재화의 판매 관련 매출장려활동이 핵심감사사항으로 결정된 이유는?
Gold IDs: ['sec1-2018-kam-1']
 → RR_avg=1.0000
   Ranked IDs: ['sec1-2018-kam-1', 'sec1-2020-kam-1', 'sec1